In [ ]:
# Visualization 4: Distribution of Predictions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (name, model) in enumerate(trained_models.items()):
    y_pred = model.predict(X_test)
    
    # Create a cross-tabulation
    pred_dist = pd.crosstab(y_test, y_pred, margins=True)
    
    axes[idx].bar(['Predicted\nSetosa', 'Predicted\nVersicolor', 'Predicted\nVirginica'],
                  [np.sum(y_pred == 0), np.sum(y_pred == 1), np.sum(y_pred == 2)])
    axes[idx].set_ylabel('Count')
    axes[idx].set_title(f'{name}\nPrediction Distribution')
    axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ All visualizations completed!")

In [ ]:
# Visualization 3: Feature Importance (for tree-based models)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Decision Tree feature importance
dt_model = trained_models['Decision Tree']
feature_importance_dt = dt_model.feature_importances_
indices_dt = np.argsort(feature_importance_dt)[::-1]

axes[0].barh(range(len(feature_importance_dt)), feature_importance_dt[indices_dt])
axes[0].set_yticks(range(len(feature_importance_dt)))
axes[0].set_yticklabels([X_scaled.columns[i] for i in indices_dt])
axes[0].set_xlabel('Importance')
axes[0].set_title('Decision Tree - Feature Importance')
axes[0].invert_yaxis()

# Random Forest feature importance
rf_model = trained_models['Random Forest']
feature_importance_rf = rf_model.feature_importances_
indices_rf = np.argsort(feature_importance_rf)[::-1]

axes[1].barh(range(len(feature_importance_rf)), feature_importance_rf[indices_rf])
axes[1].set_yticks(range(len(feature_importance_rf)))
axes[1].set_yticklabels([X_scaled.columns[i] for i in indices_rf])
axes[1].set_xlabel('Importance')
axes[1].set_title('Random Forest - Feature Importance')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Visualization 1: Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (name, model) in enumerate(trained_models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], 
                cbar=False, xticklabels=['setosa', 'versicolor', 'virginica'],
                yticklabels=['setosa', 'versicolor', 'virginica'])
    axes[idx].set_title(f'{name}\nConfusion Matrix')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

# Visualization 2: Model Performance Comparison
fig, ax = plt.subplots(figsize=(10, 6))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.25

for i, row in results_df.iterrows():
    values = [row['Accuracy'], row['Precision'], row['Recall'], row['F1-Score']]
    ax.bar(x + i*width, values, width, label=row['Model'])

ax.set_xlabel('Metrics')
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Section 8: Visualize Results

Create visualizations including confusion matrices, ROC curves, and feature importance plots to understand model behavior.

In [ ]:
# Evaluate all models
results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    })
    
    print(f"\n{name} Performance:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")

# Create a results DataFrame
results_df = pd.DataFrame(results)
print("\n" + "="*60)
print("Model Comparison:")
print("="*60)
print(results_df.to_string(index=False))

# Cross-validation scores for the best model
best_model_name = results_df.loc[results_df['F1-Score'].idxmax(), 'Model']
best_model = trained_models[best_model_name]

cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='f1_weighted')
print(f"\n5-Fold Cross-Validation Scores for {best_model_name}:")
print(f"  Scores: {cv_scores}")
print(f"  Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## Section 7: Evaluate Model Performance

Calculate metrics like accuracy, precision, recall, and F1-score to assess model performance on the test set.

In [ ]:
# Train multiple models for comparison
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=200),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=5),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100, max_depth=5)
}

trained_models = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    # Training accuracy
    train_pred = model.predict(X_train)
    train_accuracy = accuracy_score(y_train, train_pred)
    print(f"{name} - Training Accuracy: {train_accuracy:.4f}")

print("\n✓ All models trained successfully!")

## Section 6: Train a Simple Classification Model

Train a machine learning model such as Logistic Regression or Decision Tree on the training data.

In [ ]:
# Split data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")
print(f"Feature dimensions: {X_train.shape[1]}")

print("\nClass distribution in training set:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nClass distribution in testing set:")
print(pd.Series(y_test).value_counts().sort_index())

## Section 5: Split Data into Training and Testing Sets

Use train_test_split() to divide the dataset into training and testing subsets with an appropriate ratio.

In [ ]:
# Prepare features and target
X = df_cleaned.iloc[:, :-2]  # All feature columns
y = df_cleaned['target']     # Target column

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeature names:")
print(X.columns.tolist())

# Create additional engineered features
X_engineered = X.copy()
X_engineered['sepal_area'] = X_engineered['sepal length (cm)'] * X_engineered['sepal width (cm)']
X_engineered['petal_area'] = X_engineered['petal length (cm)'] * X_engineered['petal width (cm)']
X_engineered['sepal_to_petal_length'] = X_engineered['sepal length (cm)'] / (X_engineered['petal length (cm)'] + 1e-5)

print("\nEnhanced features with engineering:")
print(X_engineered.head())
print("\nNew feature shape:", X_engineered.shape)

# Feature Scaling - Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_engineered)
X_scaled = pd.DataFrame(X_scaled, columns=X_engineered.columns)

print("\nScaled Features (first 5 rows):")
print(X_scaled.head())
print("\nMean of scaled features (should be close to 0):")
print(X_scaled.mean())

## Section 4: Feature Engineering

Create new features, scale numerical features, and encode categorical variables using appropriate techniques.

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

# Check for duplicates
print(f"\nDuplicate rows: {df.duplicated().sum()}")

# Check data types
print("\nData Types:")
print(df.dtypes)

# Remove duplicates if any
df_cleaned = df.drop_duplicates().reset_index(drop=True)
print(f"\nDataset shape after cleaning: {df_cleaned.shape}")

# Check for outliers using IQR method
print("\nOutlier Detection (using IQR method):")
for column in df_cleaned.iloc[:, :-2]:  # Exclude target and target_name
    Q1 = df_cleaned[column].quantile(0.25)
    Q3 = df_cleaned[column].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df_cleaned[(df_cleaned[column] < Q1 - 1.5*IQR) | (df_cleaned[column] > Q3 + 1.5*IQR)]
    print(f"{column}: {len(outliers)} outliers found")

## Section 3: Data Preprocessing and Cleaning

Handle missing values, remove duplicates, and deal with outliers to prepare the dataset for modeling.

In [ ]:
# Load the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# Create a DataFrame for easier exploration
df = pd.DataFrame(X, columns=iris.feature_names)
df['target'] = y
df['target_name'] = df['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nStatistical Summary:")
print(df.describe())

## Section 2: Load and Explore Dataset

Load a dataset using Pandas and explore its structure, shape, data types, and summary statistics.

In [ ]:
# Import essential libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## Section 1: Import Required Libraries

Import essential libraries such as NumPy, Pandas, Scikit-learn, and Matplotlib for machine learning tasks.

# Machine Learning Learning Guide

This notebook covers the fundamentals of Machine Learning, including data exploration, preprocessing, model training, and evaluation. We'll work through a complete ML pipeline step-by-step.